In [1]:
import numpy as np
import torch

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_steps = 8
val_steps = 3
test_steps = 3
max_epochs = 5000
early_stop_epochs = 3000
lr = 0.1
sim_coeff = 1.0
deg_coeff = 1.0
reg_coeff = 0.0 # 0.001
ph_list = [0.1, 0.1, 0.1, 0.1, 0.1]
padd = 0.1
n = 100

In [3]:
# Define a function to format an array to three decimal places
def format_array(array):
    return [f"{num:.1f}" for num in array]

In [4]:
seed_list = [0, 10, 20]
def get_mean_std_deviations(initial_params_all, w_list, w_add, reg_coeff):
    H = len(ph_list)
    T = train_steps + val_steps + test_steps + 1
    ph_list_str = ''.join(str(e) for e in ph_list)
    w_list_str = ''.join(str(e) for e in w_list)
    initial_values = initial_params_all[:-1]
    initial_addition = initial_params_all[-1]
    parameters_full = np.ones((len(seed_list), len(initial_params_all)+1))
    test_cost_full = np.ones(len(seed_list))
    has_res = False
    for ind, seed in enumerate(seed_list):
        data_info = (f'simple_synthetic/n{n}_T{T}_H{H}_ph{ph_list_str}_padd{padd}_w{w_list_str}_wAdd{w_add}_seed{seed}')
        for val_ind in range(len(initial_values)):
            if initial_values[val_ind] == 0:
                initial_values[val_ind] = 0.0001 # do not have actual zero
        if initial_addition == 0:
            initial_addition = 0.0001 # do not have actual zero
        elif initial_addition == 1:
            initial_addition = 0.9999 # do not have actual one
        initial_values_str = ''.join(str(e) for e in initial_values)
        args_info = (f'trainSteps{train_steps}_val{val_steps}_test{test_steps}'
                f'_maxEpochs{max_epochs}_early{early_stop_epochs}_lr{lr}_seed{seed}'
                    f'initVal{initial_values_str}_initAdd{initial_addition}'
                    f'_simCoeff{sim_coeff}_degCoeff{deg_coeff}_regCoeff{reg_coeff}')
        
        try:
            parameters_full[ind, 1:] = np.load('../saved_parameters/'+data_info+'/'+args_info+'_parameters.npy')
            test_cost_full[ind] = np.load('../saved_epoch_cost/'+data_info+'/'+args_info+'_epoch_cost.npy')[-1]
            has_res = True
        except FileNotFoundError:
            parameters_full[ind] = np.nan
            test_cost_full[ind] = np.nan
    if has_res:
        parameters_mean = np.nanmean(parameters_full, axis=0)
        parameters_std = np.nanstd(parameters_full, axis=0)
        test_cost_mean = np.nanmean(test_cost_full)
        test_cost_std = np.nanstd(test_cost_full)

        initial_parameter_values = np.ones_like(parameters_mean)
        initial_parameter_values[1:-1] = initial_values
        initial_parameter_values[-1] = initial_addition

        deviations_from_initial = parameters_mean - initial_parameter_values
        return parameters_mean, parameters_std, deviations_from_initial, test_cost_mean, test_cost_std
    else:
        return None, None, None, None, None

#### Analysis test costs as well as parameters

In [5]:
initial_params_list = [[0.1, 0.1, 0.1, 0.1, 0.1],
                       [0.2, 0.2, 0.2, 0.2, 0.2],
                       [0.5, 0.5, 0.5, 0.5, 0.5],
                       [1.0, 1.0, 1.0, 1.0, 1.0],
                       [2.0, 2.0, 2.0, 2.0, 1.0],
                       [5.0, 5.0, 5.0, 5.0, 1.0],
                       [1.0, 0.1, 0.1, 0.1, 0.1],
                       [0.1, 1.0, 0.1, 0.1, 0.1],
                       [0.1, 0.1, 1.0, 0.1, 0.1],
                       [0.1, 0.1, 0.1, 1.0, 0.1],
                       [0.1, 0.1, 0.1, 0.1, 1.0]
]

In [6]:
true_w_list = [
    [1.0, 0.0, 1.0, 0.0, 1.0],
    [1.0, 0.0, 1.0, 1.0, 0.0],
    [1.0, 0.0, 0.6, 0.0, 1.2],
    [1.0, 1.2, 0.0, 0.0, 0.6],
    [1.0, 0.0, 1.0, 1.0, 1.0],
    [1.0, 1.0, 1.0, 1.0, 0.0],
    [1.0, 0.0, 0.6, 0.3, 1.2],
    [1.0, 0.6, 1.2, 0.0, 0.3]
]

true_w_add_list = [0.0, 0.3]

In [8]:
for w_list in true_w_list:
    for w_add in true_w_add_list:
        print(f'w_list: {w_list}, w_add: {w_add}')
        for initial_param in initial_params_list:
            parameters_mean, parameters_std, deviations_from_initial, test_cost_mean, test_cost_std = get_mean_std_deviations(initial_param, w_list, w_add, reg_coeff)
            if parameters_mean is not None:
                print(f'Initial values: {format_array(initial_param)}')
                print(f'Mean parameters: {format_array(parameters_mean)}')
                print(f'Std parameters: {format_array(parameters_std)}')
                print(f'Deviations from initial: {format_array(deviations_from_initial)}')
                print(f'Mean test cost: {test_cost_mean:.6f}')
                print(f'Std test cost: {test_cost_std:.6f}')
                print('\n')
        print('-'*100)

w_list: [1.0, 0.0, 1.0, 0.0, 1.0], w_add: 0.0
Initial values: ['1.0', '1.0', '1.0', '1.0', '1.0']
Mean parameters: ['1.0', '0.0', '1.0', '0.0', '1.0', '0.0']
Std parameters: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0']
Deviations from initial: ['0.0', '-1.0', '-0.0', '-1.0', '-0.0', '-1.0']
Mean test cost: 0.000001
Std test cost: 0.000000


Initial values: ['2.0', '2.0', '2.0', '2.0', '1.0']
Mean parameters: ['1.0', '0.0', '1.0', '0.0', '1.0', '0.0']
Std parameters: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0']
Deviations from initial: ['0.0', '-2.0', '-1.0', '-2.0', '-1.0', '-1.0']
Mean test cost: 0.000001
Std test cost: 0.000000


Initial values: ['5.0', '5.0', '5.0', '5.0', '1.0']
Mean parameters: ['1.0', '0.0', '1.0', '0.0', '1.0', '0.0']
Std parameters: ['0.0', '0.0', '0.0', '0.0', '0.0', '0.0']
Deviations from initial: ['0.0', '-5.0', '-4.0', '-5.0', '-4.0', '-1.0']
Mean test cost: 0.000001
Std test cost: 0.000000


Initial values: ['1.0', '0.1', '0.1', '0.1', '0.1']
Mean parameters: 

In [16]:
for w_list in true_w_list:
    for w_add in true_w_add_list:
        print('GT&', end='')
        for w_h in w_list[1:]:
            print(f'{w_h:.1f}&', end='')
        print(f'{w_add:.1f}\\\\')
        for reg_coeff in [0.0, 0.001]:
            print(f'$\\alpha_3={reg_coeff}$&', end='')
            best_parameters_mean = None
            best_parameters_std = 0
            best_test_cost_mean = 10000
            best_test_cost_std = 0
            for initial_param in initial_params_list:
                parameters_mean, parameters_std, deviations_from_initial, test_cost_mean, test_cost_std = get_mean_std_deviations(initial_param, w_list, w_add, reg_coeff)
                if parameters_mean is not None and test_cost_mean < best_test_cost_mean:
                    best_test_cost_mean = test_cost_mean
                    best_test_cost_std = test_cost_std
                    best_parameters_mean = parameters_mean
                    best_parameters_std = parameters_std
            if best_parameters_mean is not None:
                for ind in range(1, len(best_parameters_mean)):
                    print(f'${best_parameters_mean[ind]:.1f}\\pm{best_parameters_std[ind]:.1f}$&', end='')
                print('\\\\')
        print('\\midrule')


GT&0.0&1.0&0.0&1.0&0.0\\
$\alpha_3=0.0$&$0.0\pm0.0$&$1.0\pm0.0$&$0.0\pm0.0$&$1.0\pm0.0$&$0.0\pm0.0$&\\
$\alpha_3=0.001$&$0.0\pm0.0$&$0.9\pm0.0$&$0.2\pm0.0$&$0.7\pm0.0$&$0.0\pm0.0$&\\
\midrule
GT&0.0&1.0&0.0&1.0&0.3\\
$\alpha_3=0.0$&$0.0\pm0.0$&$1.0\pm0.0$&$0.0\pm0.0$&$1.0\pm0.0$&$0.3\pm0.0$&\\
$\alpha_3=0.001$&$0.0\pm0.0$&$0.9\pm0.0$&$0.2\pm0.0$&$0.7\pm0.0$&$0.3\pm0.0$&\\
\midrule
GT&0.0&1.0&1.0&0.0&0.0\\
$\alpha_3=0.0$&$0.0\pm0.0$&$1.0\pm0.0$&$1.0\pm0.0$&$0.0\pm0.0$&$0.0\pm0.0$&\\
$\alpha_3=0.001$&$0.0\pm0.0$&$0.9\pm0.0$&$0.8\pm0.0$&$0.1\pm0.0$&$0.0\pm0.0$&\\
\midrule
GT&0.0&1.0&1.0&0.0&0.3\\
$\alpha_3=0.0$&$0.0\pm0.0$&$1.0\pm0.0$&$1.0\pm0.0$&$0.0\pm0.0$&$0.3\pm0.0$&\\
$\alpha_3=0.001$&$0.0\pm0.0$&$0.9\pm0.0$&$0.8\pm0.0$&$0.1\pm0.0$&$0.3\pm0.0$&\\
\midrule
GT&0.0&0.6&0.0&1.2&0.0\\
$\alpha_3=0.0$&$0.0\pm0.0$&$0.6\pm0.0$&$0.0\pm0.0$&$1.2\pm0.0$&$0.0\pm0.0$&\\
$\alpha_3=0.001$&$0.0\pm0.0$&$0.5\pm0.0$&$0.1\pm0.0$&$0.9\pm0.0$&$0.0\pm0.0$&\\
\midrule
GT&0.0&0.6&0.0&1.2&0.3\\
$\alpha_3=0.0$&